# MNIST → EMNIST TRANSFER LEARNING WITH A CUSTOM CNN

This notebook demonstrates the complete transfer learning workflow using a custom convolutional neural network (CNN).

We will:

1. Train a CNN on the MNIST handwritten digit dataset
2. Save the pretrained weights
3. Transfer the learned features to EMNIST letter classification
4. Freeze the convolutional backbone
5. Fine-tune the model on the new task
6. Evaluate performance using multiple metrics

The goal is to understand:
- feature learning
- transfer learning
- freezing vs fine-tuning
- domain adaptation


# 1. IMPORTS AND SETUP

We first import all required libraries.

This includes:
- PyTorch for deep learning
- torchvision for datasets and transforms
- sklearn for evaluation metrics
- matplotlib for visualization


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import MNIST

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)


# 2. REPRODUCIBILITY AND DEVICE SETUP

Deep learning experiments can produce slightly different results due to randomness.

To improve reproducibility, we:
- seed Python random generators
- seed NumPy
- seed PyTorch

We also automatically select the best available hardware:
- CUDA (NVIDIA GPU)
- MPS (Apple Silicon GPU)
- CPU fallback


In [ ]:
def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
seed_everything()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")


# 3. CUSTOM EMNIST DATASET

The EMNIST subset is organized in folders:

```text
train/
    A/
    B/
    C/
    ...
```

Each folder corresponds to one class.

We implement a custom PyTorch Dataset class that:
- scans folders
- loads images
- converts labels to integers
- applies preprocessing transforms

This demonstrates how custom datasets are built in PyTorch.


In [ ]:
class EMNISTFolderDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and not d.startswith('.')
        ])

        self.class_to_idx = {
            cls_name: i
            for i, cls_name in enumerate(self.classes)
        }

        self.samples = []

        for cls_name in self.classes:

            cls_dir = os.path.join(root_dir, cls_name)

            for img_name in os.listdir(cls_dir):

                if img_name.endswith(".png"):

                    img_path = os.path.join(cls_dir, img_name)

                    self.samples.append(
                        (img_path, self.class_to_idx[cls_name])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("L")

        if self.transform:
            image = self.transform(image)

        return image, label


# 4. IMAGE TRANSFORMS

Neural networks require tensors as input.

We apply several preprocessing operations:
- resize images to 28×28
- convert images to tensors
- normalize pixel values

Normalization improves optimization stability during training.


In [ ]:
mnist_transforms = transforms.Compose([

    transforms.Resize((28, 28)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.1307,),
        std=(0.3081,)
    )
])


# 5. LOAD THE MNIST DATASET

We now load the MNIST dataset.

MNIST contains:
- grayscale handwritten digits
- 10 classes (0–9)

This dataset will be used for the pretraining phase.



In [ ]:
mnist_train_dataset = MNIST(
    #root="data",
    root="/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets",
    train=True,
    download=True,
    transform=mnist_transforms
)

mnist_test_dataset = MNIST(
    #root="data",
    root="/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets",
    train=False,
    download=True,
    transform=mnist_transforms
)


In [ ]:
batch_size = 128

mnist_train_loader = DataLoader(
    mnist_train_dataset,
    batch_size=batch_size,
    shuffle=True
)

mnist_test_loader = DataLoader(
    mnist_test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# 6. DEFINE A SIMPLE CNN

We define a small convolutional neural network.

The model has two main parts:

## Feature Extractor
The convolutional layers learn visual patterns such as:
- edges
- corners
- strokes
- shapes

## Classifier
The fully connected layers map learned features to output classes.

This separation becomes very important for transfer learning.


In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 7 * 7, 128),

            nn.ReLU(),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x



# 7. PRETRAINING ON MNIST

We first train the CNN on MNIST digit classification.

During this phase:
- the network learns generic visual features
- convolutional filters become useful feature detectors

These learned representations will later be reused for EMNIST letters.


In [ ]:
model = SimpleCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

epochs = 5
print("TRAINING ON MNIST")

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(mnist_train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * x.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()

        total += y.size(0)

    train_loss = total_loss / total
    train_accuracy = correct / total

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy:.4f}"
    )


# 8. SAVE PRETRAINED WEIGHTS

After training on MNIST, we save the learned model weights.

This simulates a common deep learning workflow:
- pretrain on one dataset
- reuse learned knowledge for another task

In [ ]:
#torch.save(model.state_dict(), "mnist_pretrained.pth")
torch.save(model.features.state_dict(), "mnist_features.pth")

print("\nSaved pretrained MNIST weights.")


# 9. LOAD THE EMNIST DATASET

We now load the EMNIST letters dataset.

Unlike MNIST:
- EMNIST contains letters instead of digits
- there are 26 output classes

We also split the data into:
- training set
- validation set
- test set

The validation set helps monitor generalization performance.


In [ ]:
emnist_trainval_dataset = EMNISTFolderDataset(
    "/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets/EMNIST_letters_subset/train",
    #"data/emnist_letters_subset/train",
    transform=mnist_transforms
)

emnist_test_dataset = EMNISTFolderDataset(
    "/leonardo/pub/userinternal/mcelori1/E_MNIST_datasets/EMNIST_letters_subset/test",
    #"data/emnist_letters_subset/test",
    transform=mnist_transforms
)

total_size = len(emnist_trainval_dataset)

train_size = int(0.8 * total_size)

val_size = total_size - train_size

emnist_train_dataset, emnist_val_dataset = random_split(
    emnist_trainval_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"\nEMNIST Train samples: {len(emnist_train_dataset)}")
print(f"EMNIST Validation samples: {len(emnist_val_dataset)}")
print(f"EMNIST Test samples: {len(emnist_test_dataset)}")

In [ ]:
emnist_train_loader = DataLoader(
    emnist_train_dataset,
    batch_size=batch_size,
    shuffle=True
)

emnist_val_loader = DataLoader(
    emnist_val_dataset,
    batch_size=batch_size,
    shuffle=False
)

emnist_test_loader = DataLoader(
    emnist_test_dataset,
    batch_size=batch_size,
    shuffle=False
)



# 10. LOAD THE PRETRAINED MODEL

We now initialize a new model and load the pretrained MNIST weights.

However, the output layer must change:
- MNIST has 10 classes
- EMNIST has 26 classes

Therefore:
- we keep the learned convolutional features
- we replace the final classifier layer

This is one of the key ideas in transfer learning.


In [ ]:
model = SimpleCNN(num_classes=26)
model = model.to(device)
model.features.load_state_dict(torch.load("mnist_features.pth"))


# 11. FREEZE THE FEATURE EXTRACTOR

Initially, we freeze the convolutional backbone.

This means:
- pretrained features are preserved
- only the classifier is trained

This strategy is useful because:
- pretrained features are already meaningful
- fewer parameters must be optimized
- training becomes more stable


In [ ]:
# Freeze convolutional feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Train only classifier first
for param in model.classifier.parameters():
    param.requires_grad = True


# 12. EARLY STOPPING

Deep neural networks can eventually begin to memorize the training data.

When this happens:
- training loss continues decreasing
- validation performance stops improving
- overfitting begins

Early stopping is a simple and effective regularization technique.

The idea is:
- monitor validation loss
- save the best model
- stop training when validation performance no longer improves

This prevents unnecessary training and often improves generalization.


Early stopping ensures that:
- the best-performing validation model is kept
- not necessarily the final epoch
- generalization performance is improved

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_acc = -float("inf")
        self.best_weights = None
        self.best_epoch = 0

    def step(self, model, val_loss, val_acc, epoch):

        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_acc = val_acc
            self.best_epoch = epoch+1
            self.counter = 0
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False

        self.counter += 1
        print(
            f"[EarlyStopping] Epoch {epoch+1 if epoch is not None else ''}: "
            f"No improvement → counter {self.counter}/{self.patience}"
        )
        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_weights)



# 13. EVALUATION FUNCTION

We define a reusable evaluation function.

This function computes:
- loss
- accuracy

on validation or test datasets.

Notice that:
- gradients are disabled with torch.no_grad()
- the model is switched to evaluation mode with model.eval()


In [ ]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            predictions = logits.argmax(dim=1)

            correct += (predictions == y).sum().item()

            total += y.size(0)

    avg_loss = total_loss / total

    accuracy = correct / total

    return avg_loss, accuracy


# 14. TRANSFER LEARNING ON EMNIST

We now train the model on EMNIST.

Training occurs in two stages.

## Stage 1 — Feature Extraction
Only the classifier is trained.

The pretrained convolutional layers remain frozen.

## Stage 2 — Fine-Tuning
After several epochs, we unfreeze the backbone.

This allows the pretrained features to adapt slightly to the new task.

We also use:
- a smaller learning rate for pretrained layers
- a larger learning rate for the new classifier

This is standard practice in transfer learning.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

early_stopping = EarlyStopping(patience=3)

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("TRANSFER LEARNING ON EMNIST")

epochs = 30

for epoch in range(epochs):

    # --------------------------------------------------------
    # Fine-tuning phase
    # --------------------------------------------------------

    if epoch == 3:

        print("\nUnfreezing convolutional backbone...\n")

        for param in model.features.parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam([
            {
                "params": model.classifier.parameters(),
                "lr": 1e-4
            },
            {
                "params": model.features.parameters(),
                "lr": 1e-5
            }
        ])

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    total_train_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(emnist_train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item() * x.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()

        total += y.size(0)

    train_loss = total_train_loss / total

    train_accuracy = correct / total

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_loss, val_accuracy = evaluate(
        model,
        emnist_val_loader,
        criterion, device
    )   
    
    # --------------------------------------------------------
    # Store metrics
    # --------------------------------------------------------

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # --------------------------------------------------------
    # Epoch summary
    # --------------------------------------------------------

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if early_stopping.step(model, val_loss, val_accuracy, epoch):
        print("\nEarly stopping triggered.")
        break


# ------------------------------------------------------------
# Restore best model
# ------------------------------------------------------------

early_stopping.restore(model)

print("\n================================================")
print("BEST MODEL SUMMARY")
print("================================================")
print(f"Best Epoch         : {early_stopping.best_epoch}")
print(f"Best Validation Acc: {early_stopping.best_acc:.4f}")


# 15. VISUALIZE TRAINING CURVES

We plot:
- training loss
- validation loss
- training accuracy
- validation accuracy

These plots help diagnose:
- overfitting
- underfitting
- convergence behavior


In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(epochs_range, train_losses, label="Training Loss", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_losses, label="Validation Loss", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
best_epoch = early_stopping.best_epoch
best_loss = early_stopping.best_loss
plt.scatter(best_epoch, best_loss, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

# Enhancing text and metadata
plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Cross Entropy Loss", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

# Strategic legend placement
plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
epochs_range = range(1, len(train_losses) + 1)
best_epoch = early_stopping.best_epoch
best_acc = early_stopping.best_acc

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(epochs_range, train_accuracies, label="Training Accuracy", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_accuracies, label="Validation Accuracy", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
plt.scatter(best_epoch, best_acc, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

# Enhancing text and metadata
plt.title("Training vs Validation Accuracy", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Accuracy", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

# Strategic legend placement
plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


# 16. FINAL TEST EVALUATION

After training, we evaluate the final model on the test set.

We compute:
- accuracy
- macro F1 score
- weighted F1 score

Why use F1 scores?

Accuracy alone can sometimes hide poor class-specific performance.

F1 scores provide a more balanced evaluation.


In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for x, y in emnist_test_loader:

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        predictions = logits.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            y.cpu().numpy()
        )

accuracy = np.mean(
    np.array(all_predictions) == np.array(all_labels)
)

f1_macro = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

f1_weighted = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("\n================================================")
print("FINAL TEST METRICS")
print("================================================")

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1   : {f1_weighted:.4f}")


# 17. CONFUSION MATRIX

The confusion matrix shows:
- correct predictions
- systematic mistakes between classes

This visualization is especially useful for handwritten character recognition because some letters look visually similar.

Examples:
- O vs Q
- I vs L
- C vs G


In [ ]:
class_names = [
    chr(i)
    for i in range(ord("A"), ord("Z") + 1)
]

cm = confusion_matrix(
    all_labels,
    all_predictions
)

fig, ax = plt.subplots(figsize=(10, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    xticks_rotation=90,
    ax=ax,
    colorbar=False
)

ax.grid(False)

plt.title("EMNIST Letters Confusion Matrix")
plt.show()

# 18. CLASSIFICATION REPORT

Finally, we print a detailed classification report containing:
- precision
- recall
- F1 score

for each class individually.

This helps identify:
- strong classes
- weak classes
- difficult character pairs


In [ ]:
print("\n================================================")
print("CLASSIFICATION REPORT")
print("================================================")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
))

# KEY TAKEAWAYS

This notebook demonstrates the full transfer learning pipeline:

1. Train on a source task
2. Save learned features
3. Reuse pretrained representations
4. Replace task-specific layers
5. Freeze and fine-tune selectively
6. Evaluate generalization performance

Most modern deep learning systems use this exact workflow at much larger scale.

Examples include:
- ImageNet pretrained vision models
- pretrained language models
- speech recognition systems
- foundation models